# Interpreting diagnostics

In [1]:
using QuantumFurnace, LinearAlgebra

Dephasing preserves every diagonal state. Gibbs is stationary, yet the
stationary state is nonunique. A complete tiny-system numerical kernel check
distinguishes this from a trajectory that merely ran for too short a time.

In [2]:
nonergodic = simulate_gibbs(Hermitian(Z); beta_phys=0.8, jumps=[Z],
    times=[0.0, 0.1], diagnostics=:strict)
@assert nonergodic.diagnostics.uniqueness == :nonunique
@assert nonergodic.diagnostics.checks.stationarity.status == :pass
nonergodic.convergence

(status = :not_reached_by_horizon, epsilon = 0.001, metric = :trace_distance, threshold_time = nothing, horizon = 0.1, requested_horizon = 0.1, max_time = 0.1, extensions = 0, initial_state_specific = true, worst_case_mixing = :not_established, numerical_floor = nothing, accuracy_evidence = :krylov_step_convergence_and_raw_state_checks)

A deliberately coarse Time grid preserves trace while missing KMS balance.
Decrease time_step to refine spacing and increase num_energy_bits to extend
the window, checking their effects separately. Custom prepared Time filters
additionally expose independent coherent frequency/time controls and policy.

In [3]:
p = prepare_gibbs_inputs(Hermitian(0.4X + 0.7Z); beta_phys=0.8,
    domain=TimeDomain(), time_step=0.8, num_energy_bits=3)
checks = workspace_diagnostics(Workspace(p.config, p.hamiltonian, p.jumps),
    p.config, p.hamiltonian)
@assert checks.checks.trace_preservation.status == :pass
@assert checks.checks.kms.status == :fail
checks.checks.stationarity

┌ Warning: OFT Integration Warning: Time array was not truncated.
│ Filter kernel at the ends should be small but it is: 4.8035470650040355e-9
└ @ QuantumFurnace ~/work/QuantumFurnace.jl/QuantumFurnace.jl/src/time_domain.jl:17


DiagnosticCheck{@NamedTuple{absolute::Float64, scaled::Float64, scale::Float64}, Float64}(:fail, (absolute = 0.14385700489638203, scaled = 0.13565974677307183, scale = 1.06042513212871), 1.0e-9, :matrix_free_action, :complete_action_probed_scale, :numerical, "Gibbs stationarity only; refine the filter/grid if the scaled residual fails.")

A work cap returns usable partial evidence and the last valid state. It
does not relax tolerances or relabel skipped diagnostics as successful.

In [4]:
partial = simulate_gibbs(Hermitian(0.3X + 0.7Z); beta_phys=0.8,
    times=[0.0, 1.0], max_matvecs=0, diagnostics=:quick,
    gap_options=(max_matvecs=0,))
@assert partial.convergence.status == :inconclusive
@assert !partial.trajectory.all_converged
partial.spectrum.reliability

:inconclusive

Trace distance is half the full trace norm. Finite trajectories constrain
their chosen initial states; multistart agreement is not all-mode coverage or
a certified worst-case mixing bound. Gibbs underflow can invalidate inverse
Gibbs-based checks even when direct-generator diagnostics remain available.

---

*This notebook was generated using [Literate.jl](https://github.com/fredrikekre/Literate.jl).*